In [1]:
import sys
sys.path.append("..")

import time
import joblib
import lightgbm
import pandas as pd
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.pipeline import Pipeline

from src.features import (
    build_features, get_numeric_features, CATEGORICAL_FEATURES, BINARY_FEATURES,
)
from src.models import build_lightgbm_preprocessor, build_lightgbm_model

train = pd.read_parquet("../data/processed/train.parquet")
valid = pd.read_parquet("../data/processed/valid.parquet")

X_train = train.drop(columns=["target"])
y_train = train["target"]
X_valid = valid.drop(columns=["target"])
y_valid = valid["target"]

numeric_features = get_numeric_features(build_features(train.head(1000)))

In [2]:
prep = build_lightgbm_preprocessor(numeric_features, CATEGORICAL_FEATURES, BINARY_FEATURES)

X_train_ready = prep.fit_transform(X_train)
X_valid_ready = prep.transform(X_valid)

print(X_train_ready.shape, X_valid_ready.shape)
X_train_ready.head()

(335558, 62) (120790, 62)


,loan_amnt,annual_inc,dti,delinq_2yrs,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,mths_since_last_major_derog,open_acc,pub_rec,...,fico_avg,emp_length_years,credit_history_months,loan_to_income,total_bal_to_limit,home_ownership,verification_status,purpose,addr_state,emp_length_missing
0,25000.0,85000.0,19.48,0.0,0.0,NaN,NaN,NaN,10.0,0.0,...,737.0,0.0,186,0.294118,NaN,3.0,2.0,2.0,4.0,0
1,7000.0,65000.0,14.29,0.0,0.0,NaN,NaN,NaN,7.0,0.0,...,707.0,0.0,91,0.107692,NaN,3.0,0.0,1.0,29.0,0
2,25000.0,70000.0,10.50,0.0,0.0,41.0,NaN,NaN,10.0,0.0,...,662.0,1.0,170,0.357143,0.672524,3.0,2.0,2.0,29.0,0
3,1200.0,54000.0,5.47,0.0,0.0,64.0,NaN,NaN,5.0,0.0,...,677.0,10.0,302,0.022222,NaN,2.0,0.0,2.0,39.0,0
4,10800.0,32000.0,11.63,0.0,1.0,58.0,NaN,NaN,14.0,0.0,...,682.0,6.0,155,0.337500,NaN,3.0,0.0,2.0,6.0,0


In [3]:
model = build_lightgbm_model()

start = time.time()
model.fit(
    X_train_ready, y_train,
    eval_set=[(X_valid_ready, y_valid)],
    eval_metric="auc",
    categorical_feature=CATEGORICAL_FEATURES,
    callbacks=[
        lightgbm.early_stopping(stopping_rounds=100),
        lightgbm.log_evaluation(period=100),
    ],
)
print(f"\nTraining time: {time.time() - start:.0f} seconds")
print("Best number of trees:", model.best_iteration_)

c:\Users\Administrator\loan-default-project\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.685703	valid_0's binary_logloss: 0.397654
[200]	valid_0's auc: 0.691936	valid_0's binary_logloss: 0.395136
[300]	valid_0's auc: 0.693381	valid_0's binary_logloss: 0.394576
[400]	valid_0's auc: 0.693806	valid_0's binary_logloss: 0.394519
[500]	valid_0's auc: 0.69367	valid_0's binary_logloss: 0.394623
Early stopping, best iteration is:
[404]	valid_0's auc: 0.693848	valid_0's binary_logloss: 0.394504

Training time: 16 seconds
Best number of trees: 404


In [4]:
p_train = model.predict_proba(X_train_ready)[:, 1]
p_valid = model.predict_proba(X_valid_ready)[:, 1]

baseline = joblib.load("../models/logistic_baseline.joblib")
p_valid_baseline = baseline.predict_proba(X_valid)[:, 1]

comparison = pd.DataFrame({
    "train_auc": [None, roc_auc_score(y_train, p_train)],
    "valid_auc": [roc_auc_score(y_valid, p_valid_baseline), roc_auc_score(y_valid, p_valid)],
    "valid_brier": [brier_score_loss(y_valid, p_valid_baseline), brier_score_loss(y_valid, p_valid)],
    "avg_predicted": [p_valid_baseline.mean(), p_valid.mean()],
}, index=["Logistic regression", "LightGBM"])

comparison["valid_gini"] = 2 * comparison["valid_auc"] - 1
print("Actual validation default rate:", round(y_valid.mean(), 4))
comparison.round(4)

Actual validation default rate: 0.1512


,train_auc,valid_auc,valid_brier,avg_predicted,valid_gini
Logistic regression,NaN,0.6804,0.1220,0.1371,0.3608
LightGBM,0.7537,0.6938,0.1206,0.1395,0.3877


In [5]:
importance = pd.DataFrame({
    "feature": X_train_ready.columns,
    "gain": model.booster_.feature_importance(importance_type="gain"),
})
importance["share_pct"] = (importance["gain"] / importance["gain"].sum() * 100).round(1)
importance.sort_values("gain", ascending=False).head(15)

,feature,gain,share_pct
60,addr_state,41113.751299,14.3
52,fico_avg,33432.881601,11.6
1,annual_inc,19089.992646,6.6
55,loan_to_income,15869.013263,5.5
22,acc_open_past_24mths,12967.219772,4.5
59,purpose,11396.224560,4.0
24,bc_open_to_buy,9673.395176,3.4
2,dti,9628.669863,3.3
31,mths_since_recent_bc,6659.017353,2.3
50,total_bc_limit,6338.802424,2.2


In [6]:
lightgbm_pipeline = Pipeline([
    ("prep", prep),
    ("model", model),
])

check = lightgbm_pipeline.predict_proba(X_valid)[:, 1]
print("Same predictions as before:", (abs(check - p_valid) < 1e-9).all())

joblib.dump(lightgbm_pipeline, "../models/lightgbm_v1.joblib")
print("Saved.")

Same predictions as before: True
Saved.


In [7]:
def train_and_score(params=None, categorical=CATEGORICAL_FEATURES):
    params = params or {}

    prep = build_lightgbm_preprocessor(numeric_features, categorical, BINARY_FEATURES)
    X_tr = prep.fit_transform(X_train)
    X_va = prep.transform(X_valid)

    model = build_lightgbm_model(**params)
    model.fit(
        X_tr, y_train,
        eval_set=[(X_va, y_valid)],
        eval_metric="auc",
        categorical_feature=categorical,
        callbacks=[lightgbm.early_stopping(stopping_rounds=100, verbose=False)],
    )

    scores = {
        "trees": model.best_iteration_,
        "train_auc": roc_auc_score(y_train, model.predict_proba(X_tr)[:, 1]),
        "valid_auc": roc_auc_score(y_valid, model.predict_proba(X_va)[:, 1]),
    }
    return scores, prep, model

In [8]:
import warnings
warnings.filterwarnings("ignore", category=lightgbm.basic.LGBMDeprecationWarning)

no_state = [c for c in CATEGORICAL_FEATURES if c != "addr_state"]

experiments = {
    "v1 defaults": ({}, CATEGORICAL_FEATURES),
    "cautious with categories": (
        {"cat_smooth": 100, "min_data_per_group": 1000}, CATEGORICAL_FEATURES),
    "without addr_state": ({}, no_state),
    "simpler trees": (
        {"num_leaves": 15, "min_child_samples": 500}, CATEGORICAL_FEATURES),
    "strong regularization": (
        {"min_child_samples": 1000, "reg_lambda": 10}, CATEGORICAL_FEATURES),
}

results = {}
trained = {}

for name, (params, categorical) in experiments.items():
    start = time.time()
    scores, prep_i, model_i = train_and_score(params, categorical)
    scores["seconds"] = round(time.time() - start)
    results[name] = scores
    trained[name] = (prep_i, model_i)
    print(f"Done: {name}")

results_table = pd.DataFrame(results).T
results_table["gap"] = results_table["train_auc"] - results_table["valid_auc"]
results_table.sort_values("valid_auc", ascending=False).round(4)

Done: v1 defaults
Done: cautious with categories
Done: without addr_state
Done: simpler trees
Done: strong regularization


,trees,train_auc,valid_auc,seconds,gap
strong regularization,458.0,0.7537,0.6949,23.0,0.0588
simpler trees,745.0,0.7406,0.6946,25.0,0.0460
cautious with categories,419.0,0.7510,0.6946,20.0,0.0564
v1 defaults,404.0,0.7537,0.6938,20.0,0.0599
without addr_state,475.0,0.7501,0.6923,19.0,0.0578


In [9]:
more_experiments = {
    "simpler + cautious": (
        {"num_leaves": 15, "min_child_samples": 500,
         "cat_smooth": 100, "min_data_per_group": 1000},
        CATEGORICAL_FEATURES,
    ),
    "simpler + cautious, slower learning": (
        {"num_leaves": 15, "min_child_samples": 500,
         "cat_smooth": 100, "min_data_per_group": 1000,
         "learning_rate": 0.02},
        CATEGORICAL_FEATURES,
    ),
}

for name, (params, categorical) in more_experiments.items():
    start = time.time()
    scores, prep_i, model_i = train_and_score(params, categorical)
    scores["seconds"] = round(time.time() - start)
    results[name] = scores
    trained[name] = (prep_i, model_i)
    print(f"Done: {name}")

results_table = pd.DataFrame(results).T
results_table["gap"] = results_table["train_auc"] - results_table["valid_auc"]
results_table.sort_values("valid_auc", ascending=False).round(4)

Done: simpler + cautious
Done: simpler + cautious, slower learning


,trees,train_auc,valid_auc,seconds,gap
"simpler + cautious, slower learning",2563.0,0.7519,0.6958,78.0,0.0561
simpler + cautious,813.0,0.7409,0.6951,28.0,0.0458
strong regularization,458.0,0.7537,0.6949,23.0,0.0588
simpler trees,745.0,0.7406,0.6946,25.0,0.0460
cautious with categories,419.0,0.7510,0.6946,20.0,0.0564
v1 defaults,404.0,0.7537,0.6938,20.0,0.0599
without addr_state,475.0,0.7501,0.6923,19.0,0.0578


In [10]:
best_name = results_table["valid_auc"].idxmax()
best_prep, best_model = trained[best_name]

importance = pd.DataFrame({
    "feature": best_prep.transform(X_valid.head(5)).columns,
    "gain": best_model.booster_.feature_importance(importance_type="gain"),
})
importance["share_pct"] = (importance["gain"] / importance["gain"].sum() * 100).round(1)

print("Best version:", best_name)
importance.sort_values("gain", ascending=False).head(10)

Best version: simpler + cautious, slower learning


,feature,gain,share_pct
52,fico_avg,89302.481481,12.5
60,addr_state,63285.823013,8.9
1,annual_inc,46239.388011,6.5
55,loan_to_income,39675.781407,5.6
22,acc_open_past_24mths,32375.017499,4.5
2,dti,26375.730802,3.7
59,purpose,26345.218479,3.7
24,bc_open_to_buy,23528.475788,3.3
31,mths_since_recent_bc,16991.808415,2.4
50,total_bc_limit,16332.700812,2.3


In [11]:
chosen_name = "simpler + cautious"
prep_v2, model_v2 = trained[chosen_name]

lightgbm_v2 = Pipeline([
    ("prep", prep_v2),
    ("model", model_v2),
])

p_valid_v2 = lightgbm_v2.predict_proba(X_valid)[:, 1]
print("Validation AUC:", round(roc_auc_score(y_valid, p_valid_v2), 4))

joblib.dump(lightgbm_v2, "../models/lightgbm_v2.joblib")

results_table.sort_values("valid_auc", ascending=False).round(4).to_csv(
    "../results/lightgbm_tuning.csv"
)
print("Saved model and tuning table.")

Validation AUC: 0.6951
Saved model and tuning table.
